In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


****Dummy Submission****

In [ ]:
import pandas as pd

sample = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

sample["Prediction"] = "A B C"

sample.to_csv("submission.csv", index=False)

sample.head()

# Miscellaneous

In [ ]:
****MileStone 1****

Q1,Calculate the frequency distribution of the correct  answer  (A, B, C, D, E) in train.csv. Based on your counts, what is the sum of the occurrences of the most frequent option and the least frequent option? 

In [2]:
import pandas as pd

# Load the training dataset
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

# Calculate the frequency of each correct answer
answer_counts = train["answer"].value_counts().sort_index()

print("Frequency Distribution:")
print(answer_counts)

# Find the most and least frequent counts
most_frequent = answer_counts.max()
least_frequent = answer_counts.min()

# Calculate their sum
result = most_frequent + least_frequent

print("\nMost Frequent Count :", most_frequent)
print("Least Frequent Count:", least_frequent)
print("Sum =", result)

Frequency Distribution:
answer
A    369
B    490
C    459
D    358
E    324
Name: count, dtype: int64

Most Frequent Count : 490
Least Frequent Count: 324
Sum = 814


Q2, After converting the prompt column to lowercase and removing all standard punctuation characters (using Python's string.punctuation), split the text by whitespace. What is the total number of unique words (vocabulary size) across the entire cleaned prompt column of train.csv?  

In [3]:
import string


# Create a translation table to remove punctuation
translator = str.maketrans("", "", string.punctuation)

# Clean the prompt column:
# 1. Convert to lowercase
# 2. Remove punctuation
# 3. Split into words
cleaned_words = (
    train["prompt"]
    .astype(str)
    .str.lower()
    .str.translate(translator)
    .str.split()
)

# Build the vocabulary (unique words)
vocabulary = set()

for words in cleaned_words:
    vocabulary.update(words)

# Vocabulary size
print("Vocabulary Size:", len(vocabulary))

Vocabulary Size: 859


Q3, Using the cleaned prompt from Row ID 1, filter out the standard English stop words using sklearn.feature_extraction.text.ENGLISH_STOP_WORDS. How many words are left in the prompt for Row ID 1 after filtering?

In [4]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Select Row ID = 1
prompt = train.loc[train["id"] == 1, "prompt"].iloc[0]

# Clean the text
cleaned = prompt.lower().translate(translator)

# Split into words
words = cleaned.split()

# Remove English stop words
filtered_words = [word for word in words if word not in ENGLISH_STOP_WORDS]

# Print results
print("Original Prompt:")
print(prompt)

print("\nWords after removing stop words:")
print(filtered_words)

print("\nNumber of words left:", len(filtered_words))

Original Prompt:
Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.

Words after removing stop words:
['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']

Number of words left: 13


Q4, Fit a default TfidfVectorizer(stop_words='english') on a list containing all the combined text of the prompts and options in train.csv. What is the exact total number of feature columns (vocabulary size) generated by the vectorizer?

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Combine prompt and all answer options into one text per row
combined_text = (
    train["prompt"].fillna("") + " " +
    train["A"].fillna("") + " " +
    train["B"].fillna("") + " " +
    train["C"].fillna("") + " " +
    train["D"].fillna("") + " " +
    train["E"].fillna("")
)

# Create the TF-IDF vectorizer
vectorizer = TfidfVectorizer(stop_words='english')

# Fit the vectorizer
X = vectorizer.fit_transform(combined_text)

# Vocabulary size (number of feature columns)
print("Total feature columns:", len(vectorizer.get_feature_names_out()))

# Alternative (same answer)
print("Shape:", X.shape)
print("Number of features:", X.shape[1])

Total feature columns: 2762
Shape: (2000, 2762)
Number of features: 2762


Q5, Using the TF-IDF vectorizer fitted in Question 3, calculate the cosine similarity between the prompt and option A strictly for Row ID 1. What is the resulting similarity score? (Round to 4 decimal places).

In [7]:
from sklearn.metrics.pairwise import cosine_similarity

# Select Row ID = 1
row = train.loc[train["id"] == 1].iloc[0]

# Transform the prompt and Option A using the already fitted vectorizer
prompt_vec = vectorizer.transform([row["prompt"]])
option_a_vec = vectorizer.transform([row["A"]])

# Calculate cosine similarity
similarity = cosine_similarity(prompt_vec, option_a_vec)[0][0]

print("Cosine Similarity:", round(similarity, 4))

Cosine Similarity: 0.272


Q6, Expand the logic from Question 4: For every row in train.csv, calculate the cosine similarity between the prompt and each of its 5 options .  Then calculate the percentage of instances where the option with the highest cosine similarity matches the correct answer. 

In [8]:
from sklearn.metrics.pairwise import cosine_similarity

correct_predictions = 0

# Loop through every row
for _, row in train.iterrows():

    # TF-IDF vector of the prompt
    prompt_vec = vectorizer.transform([row["prompt"]])

    similarities = []

    # Calculate similarity with each option
    for option in ["A", "B", "C", "D", "E"]:
        option_vec = vectorizer.transform([row[option]])
        sim = cosine_similarity(prompt_vec, option_vec)[0][0]
        similarities.append(sim)

    # Find the option with the highest similarity
    predicted_option = ["A", "B", "C", "D", "E"][similarities.index(max(similarities))]

    # Compare with the actual answer
    if predicted_option == row["answer"]:
        correct_predictions += 1

# Calculate percentage accuracy
percentage = (correct_predictions / len(train)) * 100

print(f"Correct Predictions : {correct_predictions}")
print(f"Total Questions     : {len(train)}")
print(f"Percentage          : {percentage:.2f}%")

Correct Predictions : 271
Total Questions     : 2000
Percentage          : 13.55%


Q7, If the ground truth answer for a question is C, what is the MAP@3 score if a model predicts C A B?

1.0

Q8, If the ground truth answer for a question is  B, what is the MAP@3 score if a model predicts D B E? 

0.5

Q9, The Majority Class Baseline: Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall MAP@3 score of this "Majority Class" baseline on train.csv?

In [9]:
# Top 3 most frequent answer labels
top3 = answer_counts.index[:3].tolist()

print("Top 3 Majority Classes:", top3)

# Calculate MAP@3
map3_score = 0

for actual in train["answer"]:
    if actual == top3[0]:
        map3_score += 1.0
    elif actual == top3[1]:
        map3_score += 0.5
    elif actual == top3[2]:
        map3_score += 1/3
    else:
        map3_score += 0

map3_score /= len(train)

print(f"Majority Class Baseline MAP@3: {map3_score:.4f}")

Top 3 Majority Classes: ['A', 'B', 'C']
Majority Class Baseline MAP@3: 0.3835


Q10, The TF-IDF Pipeline: Build a basic pipeline that evaluates every row in train.csv. For each row, calculate the TF-IDF cosine similarity between the prompt and each of the 5 options. Sort these options from highest similarity to lowest to form your top 3 predictions. What is the final average MAP@3 score of this TF-IDF pipeline across the entire training set?

In [10]:
from sklearn.metrics.pairwise import cosine_similarity

map3_score = 0

# Loop through each question
for _, row in train.iterrows():

    # TF-IDF vector of the prompt
    prompt_vec = vectorizer.transform([row["prompt"]])

    similarities = []

    # Compute similarity with each option
    for option in ["A", "B", "C", "D", "E"]:
        option_vec = vectorizer.transform([row[option]])
        sim = cosine_similarity(prompt_vec, option_vec)[0][0]
        similarities.append((option, sim))

    # Sort options by similarity (highest first)
    similarities.sort(key=lambda x: x[1], reverse=True)

    # Top 3 predicted options
    predictions = [x[0] for x in similarities[:3]]

    # Ground truth
    actual = row["answer"]

    # Calculate AP@3 for this row
    if actual in predictions:
        rank = predictions.index(actual) + 1
        map3_score += 1 / rank

# Final MAP@3
map3_score /= len(train)

print(f"TF-IDF Pipeline MAP@3: {map3_score:.4f}")

TF-IDF Pipeline MAP@3: 0.2962
